# Portfolio Manager – from weights to orders

**Goal:** turn the portfolio weights proposed by the Portfolio Manager (e.g. 40% WORLD, 20% GOLD) into at most 5 concrete orders that respect the competition rules.

This part is **plain Python, no AI**: calculations must be exact and testable. The AI agent proposes the weights; this function does the maths.

Competition rules applied (from the QTEM FAQ):
- max 5 orders per trading day (buys and sells both count)
- orders executed at the closing price of day T-1
- fractional units allowed
- proceeds of sells can fund buys on the same day
- cash can never go below zero; if cash is short, Belfius cuts the **last** buy in the list

## 1. Load the prices
Upload `prices.csv` (the team database) with the folder icon on the left, then run this cell. We use the **raw close** (`close`), which is what Belfius uses.

In [7]:
import pandas as pd

df = pd.read_csv("prices.csv")

# Convert the "date" column from text to real dates.
# errors="coerce": anything that is not a valid date becomes NaT ("Not a Time", empty date)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Show any rows whose date could not be read, then drop them
bad = df[df["date"].isna()]
if len(bad) > 0:
    print("WARNING:", len(bad), "rows with an invalid date were removed:")
    print(bad.head())
    df = df.dropna(subset=["date"])

last_date = df["date"].max()
prices = df[df["date"] == last_date].set_index("code")["close"].to_dict()
print("Prices of", last_date.date(), "for", len(prices), "ETFs")

      date             code                isin               yahoo    theme  \
33460  NaT  European equity  240.14999389648438  240.14999389648438  19098.0   

          close  adj_close  volume  return  
33460 -0.003113        NaN     NaN     NaN  
Prices of 2026-11-10 for 50 ETFs


## 2. The functions
- `portfolio_value`: cash + units × price
- `check_weights`: rejects unknown ETFs, negative weights, weights summing above 100%
- `weights_to_orders`: the core logic (see comments step by step)
- `apply_orders`: simulates the portfolio after the orders, and checks cash never goes negative

In [8]:
import math

MAX_ORDERS = 5          # competition rule: max 5 orders per trading day
MIN_TRADE_EUR = 50.0    # our choice: ignore tiny adjustments, they waste an order
DECIMALS = 4            # fractional units are allowed


def portfolio_value(portfolio, prices):
    """Cash + units held x price."""
    invested = sum(units * prices[etf] for etf, units in portfolio["units"].items())
    return portfolio["cash"] + invested


def check_weights(target_weights, prices):
    """Basic sanity checks on the weights proposed by the Portfolio Manager."""
    errors = []
    for etf, w in target_weights.items():
        if etf not in prices:
            errors.append(f"{etf}: not in the ETF list / no price")
        if w < 0:
            errors.append(f"{etf}: negative weight ({w})")
    total = sum(target_weights.values())
    if total > 1.0001:
        errors.append(f"weights sum to {total:.4f} > 1 (no borrowing allowed)")
    return errors


def weights_to_orders(portfolio, prices, target_weights,
                      max_orders=MAX_ORDERS, min_trade_eur=MIN_TRADE_EUR):
    """Turn target weights into at most `max_orders` orders.

    portfolio      : {"cash": 10000.0, "units": {"WORLD": 30.0, ...}}
    prices         : {"WORLD": 105.3, ...}  closing prices of day T-1
    target_weights : {"WORLD": 0.40, "GOLD": 0.20, ...}  (rest stays in cash)
    """
    errors = check_weights(target_weights, prices)
    if errors:
        raise ValueError("Invalid weights:\n- " + "\n- ".join(errors))

    total = portfolio_value(portfolio, prices)

    # 1. Gap between where we are and where we want to be, for every ETF
    #    sorted() = fixed alphabetical order, so results are the same every run
    etfs = sorted(set(portfolio["units"]) | set(target_weights))
    gaps = []
    for etf in etfs:
        held = portfolio["units"].get(etf, 0.0)
        current_eur = held * prices[etf]
        target_eur = target_weights.get(etf, 0.0) * total
        gap = target_eur - current_eur           # >0 buy, <0 sell
        if abs(gap) >= min_trade_eur:
            gaps.append((etf, gap))

    # 2. Rank the moves by size: the biggest gaps matter most.
    #    Amounts rounded to the cent; equal amounts -> alphabetical order (deterministic)
    gaps.sort(key=lambda x: (-round(abs(x[1]), 2), x[0]))
    sells_left = [g for g in gaps if g[1] < 0]

    # 3. Pick at most `max_orders` moves. A buy is only kept if we can pay for it:
    #    if cash is short, we first pull in the biggest remaining sell to fund it.
    chosen, cash_avail, dropped = [], portfolio["cash"], []
    for etf, gap in gaps:
        if (etf, gap) in chosen:
            continue
        if len(chosen) >= max_orders:
            dropped.append((etf, gap))
            continue
        if gap < 0:
            chosen.append((etf, gap)); sells_left.remove((etf, gap))
            cash_avail += -gap
            continue
        while cash_avail < gap and sells_left and len(chosen) < max_orders - 1:
            s = sells_left.pop(0)
            chosen.append(s); cash_avail += -s[1]
        amount = min(gap, cash_avail)
        if amount < min_trade_eur:
            dropped.append((etf, gap))
            continue
        chosen.append((etf, amount)); cash_avail -= amount

    # 4. Build the orders: sells first (they free up cash), then buys, biggest first
    orders, cash = [], portfolio["cash"]
    for etf, gap in chosen:
        if gap < 0:
            held = portfolio["units"].get(etf, 0.0)
            if target_weights.get(etf, 0.0) == 0:
                units = held                      # full exit: sell everything
            else:
                units = min(held, round(-gap / prices[etf], DECIMALS))
            cash += units * prices[etf]
            orders.append({"action": "SELL", "etf": etf, "units": units,
                           "price": prices[etf], "amount_eur": round(units * prices[etf], 2)})
    for etf, amount in sorted([c for c in chosen if c[1] > 0], key=lambda x: x[1], reverse=True):
        amount = min(amount, cash)
        units = math.floor(amount / prices[etf] * 10**DECIMALS) / 10**DECIMALS  # round down
        cash -= units * prices[etf]
        orders.append({"action": "BUY", "etf": etf, "units": units,
                       "price": prices[etf], "amount_eur": round(units * prices[etf], 2)})

    return {"orders": orders, "cash_after": round(cash, 2),
            "portfolio_value": round(total, 2), "not_executed": dropped}


def apply_orders(portfolio, orders):
    """Simulate the portfolio after the orders (to check the result)."""
    new = {"cash": portfolio["cash"], "units": dict(portfolio["units"])}
    for o in orders:
        sign = 1 if o["action"] == "BUY" else -1
        new["units"][o["etf"]] = new["units"].get(o["etf"], 0.0) + sign * o["units"]
        new["cash"] -= sign * o["units"] * o["price"]
        if abs(new["units"][o["etf"]]) < 1e-9:
            del new["units"][o["etf"]]
    assert new["cash"] >= -1e-6, "cash went negative!"
    return new

## 3. Test 1 – first trading day (all cash)

In [9]:
portfolio = {"cash": 10000.0, "units": {}}
weights_day1 = {"WORLD": 0.40, "SEMIS": 0.20, "GOLD": 0.15,
                "EUR GOVT BOND": 0.15, "OVERNIGHT CASH": 0.10}

result = weights_to_orders(portfolio, prices, weights_day1)
display(pd.DataFrame(result["orders"]))
print("Cash left:", result["cash_after"])
portfolio = apply_orders(portfolio, result["orders"])

,action,etf,units,price,amount_eur
0,BUY,WORLD,31.1187,128.539993,4000.00
1,BUY,SEMIS,135.1351,14.800000,2000.00
2,BUY,EUR GOVT BOND,14.1429,106.059998,1500.00
3,BUY,GOLD,21.3553,70.239998,1500.00
4,BUY,OVERNIGHT CASH,9.0560,110.422997,999.99


Cash left: 0.02


## 4. Test 2 – rebalancing with more than 5 changes
Seven changes are needed, but only 5 orders are allowed: the function keeps the biggest ones and makes sure every buy is funded.

In [10]:
weights_day2 = {"WORLD": 0.30, "SEMIS": 0.10, "GOLD": 0.10, "DEFENSE": 0.15,
                "S&P 500": 0.15, "INDIA": 0.10, "OVERNIGHT CASH": 0.10}

result = weights_to_orders(portfolio, prices, weights_day2)
display(pd.DataFrame(result["orders"]))
print("Cash left:", result["cash_after"])
print("Not executed (for the next trading day):", result["not_executed"])
portfolio = apply_orders(portfolio, result["orders"])
print(portfolio)

,action,etf,units,price,amount_eur
0,SELL,EUR GOVT BOND,14.1429,106.059998,1500.00
1,SELL,SEMIS,67.5675,14.800000,1000.00
2,SELL,WORLD,7.7797,128.539993,1000.00
3,BUY,DEFENSE,29.4002,51.020000,1500.00
4,BUY,S&P 500,2.0689,725.000000,1499.95


Cash left: 500.07
Not executed (for the next trading day): [('INDIA', 1000.0), ('GOLD', -499.9962263801574)]
{'cash': 500.0670034748082, 'units': {'WORLD': 23.339, 'SEMIS': 67.5676, 'GOLD': 21.3553, 'OVERNIGHT CASH': 9.056, 'DEFENSE': 29.4002, 'S&P 500': 2.0689}}


## 5. Test 3 – invalid weights are rejected

In [11]:
try:
    weights_to_orders(portfolio, prices, {"WORLD": 0.8, "GOLD": 0.5, "XYZ": 0.1})
except ValueError as e:
    print(e)

Invalid weights:
- XYZ: not in the ETF list / no price
- weights sum to 1.4000 > 1 (no borrowing allowed)
